In [1]:
# """
# Extract CNN/ViT feature vectors for all model × dataset combinations.
# Saves each as a .npz file: features (N, D) + labels (N,) as strings.
# Skips existing files — safely resumable if interrupted.

# Requirements:
#     pip install transformers datasets tqdm torch pillow
# """

# import os
# import numpy as np
# import torch
# from transformers import AutoImageProcessor, AutoModel, CLIPVisionModel, CLIPImageProcessor
# from datasets import load_dataset
# from tqdm import tqdm

# # ---------------------------------------------------------------------------
# # Config — edit these if needed
# # ---------------------------------------------------------------------------
# OUTPUT_DIR  = "./features"   # where .npz files are saved
# MAX_SAMPLES = 5000           # images per dataset; set to 0 for all
# BATCH_SIZE  = 64
# # ---------------------------------------------------------------------------

# os.makedirs(OUTPUT_DIR, exist_ok=True)
# DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# print(f"Device: {DEVICE}")


# def pool(outputs):
#     """pooler_output: squeeze (B,C,1,1) → (B,C) or pass through (B,C)."""
#     p = outputs.pooler_output
#     return p.squeeze(-1).squeeze(-1) if p.dim() == 4 else p

# def cls_token(outputs):
#     """CLS token from last_hidden_state — used for DINOv2."""
#     return outputs.last_hidden_state[:, 0]


# # ---------------------------------------------------------------------------
# # Models
# # Training paradigm spread:
# #   ResNet-50/101    supervised CNN        ImageNet-1k
# #   EfficientNet-B4  supervised CNN        ImageNet-1k
# #   ViT-B/16         supervised ViT        ImageNet-21k
# #   Swin-B           supervised ViT        ImageNet-22k
# #   DINOv2-B         self-supervised ViT   no classification labels
# #   CLIP ViT-B/32    vision-language ViT   web image-text pairs
# # ---------------------------------------------------------------------------
# MODEL_CONFIGS = [
#     {"id": "microsoft/resnet-50",                    "name": "resnet50",        "loader": "auto", "feature_fn": pool},
#     {"id": "microsoft/resnet-101",                   "name": "resnet101",       "loader": "auto", "feature_fn": pool},
#     {"id": "google/efficientnet-b4",                 "name": "efficientnet_b4", "loader": "auto", "feature_fn": pool},
#     {"id": "google/vit-base-patch16-224",            "name": "vit_b16",         "loader": "auto", "feature_fn": cls_token},  # pooler missing from checkpoint; CLS token is correct
#     {"id": "microsoft/swin-base-patch4-window7-224", "name": "swin_b",          "loader": "auto", "feature_fn": pool},
#     {"id": "facebook/dinov2-base",                   "name": "dinov2_b",        "loader": "auto", "feature_fn": cls_token},
#     {"id": "openai/clip-vit-base-patch32",           "name": "clip_vit_b32",    "loader": "clip", "feature_fn": pool},
# ]

# # ---------------------------------------------------------------------------
# # Datasets
# # Domain spread:
# #   cifar10         10 coarse objects         in-distribution baseline
# #   cifar100        100 fine-grained objects  partially in-distribution
# #   oxford_pets     37 cat/dog breeds         fine-grained, partially in ImageNet-1k
# #   food101         101 food categories       in ImageNet-1k but very different visuals
# #   flowers102      102 flower species        mostly OOD
# #   beans           3 plant health classes    OOD — agricultural domain
# #   fashion_mnist   10 clothing types         OOD — grayscale, rendered, no natural scenes
# #   mini_imagenet   100 ImageNet classes      clean in-distribution baseline at higher res
# # ---------------------------------------------------------------------------
# DATASET_CONFIGS = [
#     {"id": "uoft-cs/cifar10",                "name": "cifar10",       "split": "test",       "image_col": "img",   "label_col": "label"},
#     {"id": "uoft-cs/cifar100",               "name": "cifar100",      "split": "test",       "image_col": "img",   "label_col": "fine_label"},
#     {"id": "timm/oxford-iiit-pet",           "name": "oxford_pets",   "split": "test",       "image_col": "image", "label_col": "label"},
#     {"id": "ethz/food101",                   "name": "food101",       "split": "validation", "image_col": "image", "label_col": "label"},
#     {"id": "dpdl-benchmark/oxford_flowers102","name": "flowers102",   "split": "test",       "image_col": "image", "label_col": "label"},
#     {"id": "AI-Lab-Makerere/beans",          "name": "beans",         "split": "test",       "image_col": "image", "label_col": "labels"},
#     {"id": "zalando-datasets/fashion_mnist", "name": "fashion_mnist", "split": "test",       "image_col": "image", "label_col": "label"},
#     {"id": "timm/mini-imagenet",             "name": "mini_imagenet", "split": "validation", "image_col": "image", "label_col": "label"},
# ]


# def load_model(cfg):
#     if cfg["loader"] == "clip":
#         processor = CLIPImageProcessor.from_pretrained(cfg["id"])
#         model     = CLIPVisionModel.from_pretrained(cfg["id"]).to(DEVICE)
#     else:
#         processor = AutoImageProcessor.from_pretrained(cfg["id"])
#         model     = AutoModel.from_pretrained(cfg["id"]).to(DEVICE)
#     model.eval()
#     return model, processor


# def extract(model, processor, feature_fn, dataset, img_col, label_col):
#     n = len(dataset)
#     if MAX_SAMPLES and n > MAX_SAMPLES:
#         rng     = np.random.default_rng(42)
#         indices = rng.choice(n, size=MAX_SAMPLES, replace=False).tolist()
#         dataset = dataset.select(indices)

#     class_names  = dataset.features[label_col].names
#     all_features = []
#     all_labels   = []

#     for start in tqdm(range(0, len(dataset), BATCH_SIZE), leave=False):
#         batch   = dataset[start : start + BATCH_SIZE]
#         images  = [img.convert("RGB") for img in batch[img_col]]
#         inputs  = processor(images=images, return_tensors="pt").to(DEVICE)
#         with torch.no_grad():
#             outputs = model(**inputs)
#         all_features.append(feature_fn(outputs).cpu().float().numpy())
#         all_labels.extend(batch[label_col])

#     features = np.concatenate(all_features, axis=0)
#     labels   = np.array([class_names[l] for l in all_labels])
#     return features, labels


# # ---------------------------------------------------------------------------
# # Main loop
# # ---------------------------------------------------------------------------
# total = len(MODEL_CONFIGS) * len(DATASET_CONFIGS)
# done  = 0

# for m_cfg in MODEL_CONFIGS:
#     print(f"\n{'='*60}\nModel: {m_cfg['name']}  ({m_cfg['id']})\n{'='*60}")

#     try:
#         model, processor = load_model(m_cfg)
#     except Exception as e:
#         print(f"  [SKIP] Could not load model: {e}")
#         done += len(DATASET_CONFIGS)
#         continue

#     for d_cfg in DATASET_CONFIGS:
#         done += 1
#         out_path = os.path.join(OUTPUT_DIR, f"{m_cfg['name']}__{d_cfg['name']}.npz")

#         if os.path.exists(out_path):
#             print(f"  [{done}/{total}] SKIP (exists): {os.path.basename(out_path)}")
#             continue

#         print(f"  [{done}/{total}] {m_cfg['name']} × {d_cfg['name']}", end=" ... ", flush=True)

#         try:
#             dataset = load_dataset(d_cfg["id"], split=d_cfg["split"])
#             features, labels = extract(model, processor, m_cfg["feature_fn"],
#                                        dataset, d_cfg["image_col"], d_cfg["label_col"])
#             np.savez_compressed(out_path, features=features, labels=labels,
#                                 model_id=m_cfg["id"], dataset_id=d_cfg["id"])
#             print(f"saved {features.shape}, {len(np.unique(labels))} classes")
#         except Exception as e:
#             print(f"FAILED: {e}")

# print("\nAll done.")

## with images:

In [2]:
# """
# Extract CNN/ViT feature vectors for all model × dataset combinations.
# Saves each as a .npz file: features (N, D) + labels (N,) as strings.
# Also saves images (N, 128, 128, 3) uint8 once per dataset in images__<dataset>.npz.

# Skips existing files — safely resumable if interrupted.
# Feature files and image files are independent: re-running only fills in what's missing.

# Requirements:
#     pip install transformers datasets tqdm torch pillow
# """

# import os
# import numpy as np
# import torch
# from transformers import AutoImageProcessor, AutoModel, CLIPVisionModel, CLIPImageProcessor
# from datasets import load_dataset
# from tqdm import tqdm

# # ---------------------------------------------------------------------------
# # Config
# # ---------------------------------------------------------------------------
# OUTPUT_DIR   = "./features"
# MAX_SAMPLES  = 5000
# BATCH_SIZE   = 64
# THUMB_SIZE   = 128          # saved image resolution (square)
# # ---------------------------------------------------------------------------

# os.makedirs(OUTPUT_DIR, exist_ok=True)
# DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# print(f"Device: {DEVICE}")


# def pool(outputs):
#     p = outputs.pooler_output
#     return p.squeeze(-1).squeeze(-1) if p.dim() == 4 else p

# def cls_token(outputs):
#     return outputs.last_hidden_state[:, 0]


# MODEL_CONFIGS = [
#     {"id": "microsoft/resnet-50",                    "name": "resnet50",        "loader": "auto", "feature_fn": pool},
#     {"id": "microsoft/resnet-101",                   "name": "resnet101",       "loader": "auto", "feature_fn": pool},
#     {"id": "google/efficientnet-b4",                 "name": "efficientnet_b4", "loader": "auto", "feature_fn": pool},
#     {"id": "google/vit-base-patch16-224",            "name": "vit_b16",         "loader": "auto", "feature_fn": cls_token},
#     {"id": "microsoft/swin-base-patch4-window7-224", "name": "swin_b",          "loader": "auto", "feature_fn": pool},
#     {"id": "facebook/dinov2-base",                   "name": "dinov2_b",        "loader": "auto", "feature_fn": cls_token},
#     {"id": "openai/clip-vit-base-patch32",           "name": "clip_vit_b32",    "loader": "clip", "feature_fn": pool},
# ]

# DATASET_CONFIGS = [
#     {"id": "uoft-cs/cifar10",                 "name": "cifar10",       "split": "test",       "image_col": "img",   "label_col": "label"},
#     {"id": "uoft-cs/cifar100",                "name": "cifar100",      "split": "test",       "image_col": "img",   "label_col": "fine_label"},
#     {"id": "timm/oxford-iiit-pet",            "name": "oxford_pets",   "split": "test",       "image_col": "image", "label_col": "label"},
#     {"id": "ethz/food101",                    "name": "food101",       "split": "validation", "image_col": "image", "label_col": "label"},
#     {"id": "dpdl-benchmark/oxford_flowers102","name": "flowers102",    "split": "test",       "image_col": "image", "label_col": "label"},
#     {"id": "AI-Lab-Makerere/beans",           "name": "beans",         "split": "test",       "image_col": "image", "label_col": "labels"},
#     {"id": "zalando-datasets/fashion_mnist",  "name": "fashion_mnist", "split": "test",       "image_col": "image", "label_col": "label"},
#     {"id": "timm/mini-imagenet",              "name": "mini_imagenet", "split": "validation", "image_col": "image", "label_col": "label"},
# ]


# def load_model(cfg):
#     if cfg["loader"] == "clip":
#         processor = CLIPImageProcessor.from_pretrained(cfg["id"])
#         model     = CLIPVisionModel.from_pretrained(cfg["id"]).to(DEVICE)
#     else:
#         processor = AutoImageProcessor.from_pretrained(cfg["id"])
#         model     = AutoModel.from_pretrained(cfg["id"]).to(DEVICE)
#     model.eval()
#     return model, processor


# def subsample(dataset, label_col):
#     """Subsample dataset to MAX_SAMPLES with fixed seed. Returns dataset."""
#     n = len(dataset)
#     if MAX_SAMPLES and n > MAX_SAMPLES:
#         indices = np.random.default_rng(42).choice(n, size=MAX_SAMPLES, replace=False).tolist()
#         dataset = dataset.select(indices)
#     return dataset


# def extract_features(model, processor, feature_fn, dataset, img_col, label_col):
#     class_names  = dataset.features[label_col].names
#     all_features = []
#     all_labels   = []

#     for start in tqdm(range(0, len(dataset), BATCH_SIZE), leave=False):
#         batch  = dataset[start : start + BATCH_SIZE]
#         images = [img.convert("RGB") for img in batch[img_col]]
#         inputs = processor(images=images, return_tensors="pt").to(DEVICE)
#         with torch.no_grad():
#             outputs = model(**inputs)
#         all_features.append(feature_fn(outputs).cpu().float().numpy())
#         all_labels.extend(batch[label_col])

#     features = np.concatenate(all_features, axis=0)
#     labels   = np.array([class_names[l] for l in all_labels])
#     return features, labels


# def extract_images(dataset, img_col, label_col):
#     """Resize all images to THUMB_SIZE and return as uint8 array (N, H, W, 3)."""
#     class_names = dataset.features[label_col].names
#     all_images  = []
#     all_labels  = []

#     for start in tqdm(range(0, len(dataset), BATCH_SIZE), leave=False):
#         batch  = dataset[start : start + BATCH_SIZE]
#         for img, lbl in zip(batch[img_col], batch[label_col]):
#             thumb = img.convert("RGB").resize((THUMB_SIZE, THUMB_SIZE))
#             all_images.append(np.array(thumb, dtype=np.uint8))
#             all_labels.append(lbl)

#     images = np.stack(all_images, axis=0)                          # (N, 128, 128, 3)
#     labels = np.array([class_names[l] for l in all_labels])
#     return images, labels


# # ---------------------------------------------------------------------------
# # Main loop
# # ---------------------------------------------------------------------------
# total = len(MODEL_CONFIGS) * len(DATASET_CONFIGS)
# done  = 0

# for m_cfg in MODEL_CONFIGS:
#     print(f"\n{'='*60}\nModel: {m_cfg['name']}  ({m_cfg['id']})\n{'='*60}")

#     # Track whether we actually need this model for any dataset
#     needs_model = any(
#         not os.path.exists(os.path.join(OUTPUT_DIR, f"{m_cfg['name']}__{d['name']}.npz"))
#         for d in DATASET_CONFIGS
#     )

#     model, processor = None, None

#     for d_cfg in DATASET_CONFIGS:
#         done += 1
#         feat_path  = os.path.join(OUTPUT_DIR, f"{m_cfg['name']}__{d_cfg['name']}.npz")
#         img_path   = os.path.join(OUTPUT_DIR, f"images__{d_cfg['name']}.npz")
#         feat_done  = os.path.exists(feat_path)
#         img_done   = os.path.exists(img_path)

#         if feat_done and img_done:
#             print(f"  [{done}/{total}] SKIP (both exist): {m_cfg['name']} × {d_cfg['name']}")
#             continue

#         # Load dataset (needed for either features or images)
#         try:
#             dataset = load_dataset(d_cfg["id"], split=d_cfg["split"])
#             dataset = subsample(dataset, d_cfg["label_col"])
#         except Exception as e:
#             print(f"  [{done}/{total}] FAILED loading dataset {d_cfg['name']}: {e}")
#             continue

#         # --- Feature vectors ---
#         if not feat_done:
#             # Lazy-load model only when first needed
#             if model is None:
#                 try:
#                     model, processor = load_model(m_cfg)
#                 except Exception as e:
#                     print(f"  [SKIP] Could not load model {m_cfg['name']}: {e}")
#                     break   # skip all datasets for this model

#             print(f"  [{done}/{total}] {m_cfg['name']} × {d_cfg['name']} features", end=" ... ", flush=True)
#             try:
#                 features, labels = extract_features(
#                     model, processor, m_cfg["feature_fn"],
#                     dataset, d_cfg["image_col"], d_cfg["label_col"]
#                 )
#                 np.savez_compressed(feat_path, features=features, labels=labels,
#                                     model_id=m_cfg["id"], dataset_id=d_cfg["id"])
#                 print(f"saved {features.shape}, {len(np.unique(labels))} classes")
#             except Exception as e:
#                 print(f"FAILED: {e}")
#         else:
#             print(f"  [{done}/{total}] SKIP features (exists): {m_cfg['name']} × {d_cfg['name']}")

#         # --- Images (once per dataset, model-independent) ---
#         if not img_done:
#             print(f"  images__{d_cfg['name']}", end=" ... ", flush=True)
#             try:
#                 images, labels = extract_images(dataset, d_cfg["image_col"], d_cfg["label_col"])
#                 np.savez_compressed(img_path, images=images, labels=labels,
#                                     dataset_id=d_cfg["id"])
#                 print(f"saved {images.shape}")
#             except Exception as e:
#                 print(f"FAILED: {e}")
#         else:
#             print(f"         SKIP images  (exists): images__{d_cfg['name']}")

# print("\nAll done.")

In [ ]:
"""
Extract CNN/ViT feature vectors for all model × dataset combinations.
Saves each as a .npz file: features (N, D) + labels (N,) as strings.
Also saves images (N, 128, 128, 3) uint8 once per dataset in images__<dataset>.npz.

Skips existing files — safely resumable if interrupted.
Feature files and image files are independent: re-running only fills in what's missing.

Requirements:
    pip install transformers datasets tqdm torch pillow
"""

import os
import numpy as np
import torch
from transformers import AutoImageProcessor, AutoModel, CLIPVisionModel, CLIPImageProcessor
from datasets import load_dataset
from tqdm import tqdm

from huggingface_hub import login
login(token="...")


# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------
OUTPUT_DIR   = "./features"
MAX_SAMPLES  = 5000
BATCH_SIZE   = 64
THUMB_SIZE   = 128          # saved image resolution (square)
# ---------------------------------------------------------------------------

os.makedirs(OUTPUT_DIR, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")


def pool(outputs):
    p = outputs.pooler_output
    return p.squeeze(-1).squeeze(-1) if p.dim() == 4 else p

def cls_token(outputs):
    return outputs.last_hidden_state[:, 0]


MODEL_CONFIGS = [
    {"id": "microsoft/resnet-50",                    "name": "resnet50",        "loader": "auto", "feature_fn": pool},
    {"id": "microsoft/resnet-101",                   "name": "resnet101",       "loader": "auto", "feature_fn": pool},
    {"id": "google/efficientnet-b4",                 "name": "efficientnet_b4", "loader": "auto", "feature_fn": pool},
    {"id": "google/vit-base-patch16-224",            "name": "vit_b16",         "loader": "auto", "feature_fn": cls_token},
    {"id": "microsoft/swin-base-patch4-window7-224", "name": "swin_b",          "loader": "auto", "feature_fn": pool},
    {"id": "facebook/dinov2-base",                   "name": "dinov2_b",        "loader": "auto", "feature_fn": cls_token},
    {"id": "openai/clip-vit-base-patch32",           "name": "clip_vit_b32",    "loader": "clip", "feature_fn": pool},
]

DATASET_CONFIGS = [
    # --- in-distribution / partial overlap ---
    {"id": "uoft-cs/cifar10",                 "name": "cifar10",       "split": "test",       "image_col": "img",   "label_col": "label"},
    {"id": "uoft-cs/cifar100",                "name": "cifar100",      "split": "test",       "image_col": "img",   "label_col": "fine_label"},
    {"id": "timm/oxford-iiit-pet",            "name": "oxford_pets",   "split": "test",       "image_col": "image", "label_col": "label"},
    {"id": "ethz/food101",                    "name": "food101",       "split": "validation", "image_col": "image", "label_col": "label"},
    {"id": "dpdl-benchmark/oxford_flowers102","name": "flowers102",    "split": "test",       "image_col": "image", "label_col": "label"},
    {"id": "AI-Lab-Makerere/beans",           "name": "beans",         "split": "test",       "image_col": "image", "label_col": "labels"},
    {"id": "zalando-datasets/fashion_mnist",  "name": "fashion_mnist", "split": "test",       "image_col": "image", "label_col": "label"},
    {"id": "timm/mini-imagenet",              "name": "mini_imagenet", "split": "validation", "image_col": "image", "label_col": "label"},
    # --- truly OOD ---
    # satellite: top-down land-use, no natural objects
    {"id": "timm/eurosat-rgb",                "name": "eurosat",        "split": "test",  "image_col": "image", "label_col": "label"},
    # medical: chest X-rays, grayscale, 2 classes (normal / pneumonia)
    {"id": "hf-vision/chest-xray-pneumonia",  "name": "chest_xray",    "split": "test",  "image_col": "image", "label_col": "label"},
    # histopathology: colorectal tissue microscopy, 9 tissue types
    {"id": "Trickxter/PathMNIST",             "name": "pathmnist",     "split": "test",  "image_col": "image", "label_col": "label"},
    # dermoscopy: close-up skin lesion photos, 7 diagnostic classes
    # NOTE: HAM10000 uses "dx" not "label" as the diagnosis column
    {"id": "marmal88/skin_cancer",            "name": "skin_cancer",   "split": "test",  "image_col": "image", "label_col": "dx"},
]


def load_model(cfg):
    if cfg["loader"] == "clip":
        processor = CLIPImageProcessor.from_pretrained(cfg["id"])
        model     = CLIPVisionModel.from_pretrained(cfg["id"]).to(DEVICE)
    else:
        processor = AutoImageProcessor.from_pretrained(cfg["id"])
        model     = AutoModel.from_pretrained(cfg["id"]).to(DEVICE)
    model.eval()
    return model, processor


def subsample(dataset, label_col):
    """Subsample dataset to MAX_SAMPLES with fixed seed. Returns dataset."""
    n = len(dataset)
    if MAX_SAMPLES and n > MAX_SAMPLES:
        indices = np.random.default_rng(42).choice(n, size=MAX_SAMPLES, replace=False).tolist()
        dataset = dataset.select(indices)
    return dataset


def to_str(lbl, class_names):
    """Convert a label to string regardless of whether it arrives as int or string."""
    if class_names is not None and isinstance(lbl, int):
        return class_names[lbl]
    return str(lbl)   # already a string, or plain-string column


def get_class_names(dataset, label_col):
    """Works for ClassLabel (has .names) and plain string columns."""
    feat = dataset.features[label_col]
    if hasattr(feat, "names"):
        return feat.names          # ClassLabel → list of strings
    return None                    # plain string column — labels already strings


def extract_features(model, processor, feature_fn, dataset, img_col, label_col):
    class_names  = get_class_names(dataset, label_col)
    all_features = []
    all_labels   = []

    for start in tqdm(range(0, len(dataset), BATCH_SIZE), leave=False):
        batch  = dataset[start : start + BATCH_SIZE]
        images = [img.convert("RGB") for img in batch[img_col]]
        inputs = processor(images=images, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            outputs = model(**inputs)
        all_features.append(feature_fn(outputs).cpu().float().numpy())
        all_labels.extend([to_str(l, class_names) for l in batch[label_col]])

    features = np.concatenate(all_features, axis=0)
    labels   = np.array(all_labels)
    return features, labels


def extract_images(dataset, img_col, label_col):
    """Resize all images to THUMB_SIZE and return as uint8 array (N, H, W, 3)."""
    class_names = get_class_names(dataset, label_col)
    all_images  = []
    all_labels  = []

    for start in tqdm(range(0, len(dataset), BATCH_SIZE), leave=False):
        batch  = dataset[start : start + BATCH_SIZE]
        for img, lbl in zip(batch[img_col], batch[label_col]):
            thumb = img.convert("RGB").resize((THUMB_SIZE, THUMB_SIZE))
            all_images.append(np.array(thumb, dtype=np.uint8))
            all_labels.append(to_str(lbl, class_names))

    images = np.stack(all_images, axis=0)                          # (N, 128, 128, 3)
    labels = np.array(all_labels)
    return images, labels


# ---------------------------------------------------------------------------
# Main loop
# ---------------------------------------------------------------------------
total = len(MODEL_CONFIGS) * len(DATASET_CONFIGS)
done  = 0

for m_cfg in MODEL_CONFIGS:
    print(f"\n{'='*60}\nModel: {m_cfg['name']}  ({m_cfg['id']})\n{'='*60}")

    # Track whether we actually need this model for any dataset
    needs_model = any(
        not os.path.exists(os.path.join(OUTPUT_DIR, f"{m_cfg['name']}__{d['name']}.npz"))
        for d in DATASET_CONFIGS
    )

    model, processor = None, None

    for d_cfg in DATASET_CONFIGS:
        done += 1
        feat_path  = os.path.join(OUTPUT_DIR, f"{m_cfg['name']}__{d_cfg['name']}.npz")
        img_path   = os.path.join(OUTPUT_DIR, f"images__{d_cfg['name']}.npz")
        feat_done  = os.path.exists(feat_path)
        img_done   = os.path.exists(img_path)

        if feat_done and img_done:
            print(f"  [{done}/{total}] SKIP (both exist): {m_cfg['name']} × {d_cfg['name']}")
            continue

        # Load dataset (needed for either features or images)
        try:
            dataset = load_dataset(d_cfg["id"], split=d_cfg["split"])
            dataset = subsample(dataset, d_cfg["label_col"])
        except Exception as e:
            print(f"  [{done}/{total}] FAILED loading dataset {d_cfg['name']}: {e}")
            continue

        # --- Feature vectors ---
        if not feat_done:
            # Lazy-load model only when first needed
            if model is None:
                try:
                    model, processor = load_model(m_cfg)
                except Exception as e:
                    print(f"  [SKIP] Could not load model {m_cfg['name']}: {e}")
                    break   # skip all datasets for this model

            print(f"  [{done}/{total}] {m_cfg['name']} × {d_cfg['name']} features", end=" ... ", flush=True)
            try:
                features, labels = extract_features(
                    model, processor, m_cfg["feature_fn"],
                    dataset, d_cfg["image_col"], d_cfg["label_col"]
                )
                np.savez_compressed(feat_path, features=features, labels=labels,
                                    model_id=m_cfg["id"], dataset_id=d_cfg["id"])
                print(f"saved {features.shape}, {len(np.unique(labels))} classes")
            except Exception as e:
                print(f"FAILED: {e}")
        else:
            print(f"  [{done}/{total}] SKIP features (exists): {m_cfg['name']} × {d_cfg['name']}")

        # --- Images (once per dataset, model-independent) ---
        if not img_done:
            print(f"  images__{d_cfg['name']}", end=" ... ", flush=True)
            try:
                images, labels = extract_images(dataset, d_cfg["image_col"], d_cfg["label_col"])
                np.savez_compressed(img_path, images=images, labels=labels,
                                    dataset_id=d_cfg["id"])
                print(f"saved {images.shape}")
            except Exception as e:
                print(f"FAILED: {e}")
        else:
            print(f"         SKIP images  (exists): images__{d_cfg['name']}")

print("\nAll done.")

/home/bhed/EVA_DS/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu

Model: resnet50  (microsoft/resnet-50)
  [1/63] SKIP (both exist): resnet50 × cifar10
  [2/63] SKIP (both exist): resnet50 × cifar100
  [3/63] SKIP (both exist): resnet50 × oxford_pets
  [4/63] SKIP (both exist): resnet50 × food101
  [5/63] SKIP (both exist): resnet50 × flowers102
  [6/63] SKIP (both exist): resnet50 × beans
  [7/63] SKIP (both exist): resnet50 × fashion_mnist
  [8/63] SKIP (both exist): resnet50 × mini_imagenet
  [9/63] SKIP (both exist): resnet50 × eurosat

Model: resnet101  (microsoft/resnet-101)


Loading weights: 100%|██████████| 624/624 [00:00<00:00, 11990.92it/s]
[transformers] ResNetModel LOAD REPORT from: microsoft/resnet-101
Key                 | Status     |  | 
--------------------+------------+--+-
classifier.1.bias   | UNEXPECTED |  | 
classifier.1.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  [10/63] resnet101 × cifar10 features ... 

KeyboardInterrupt: 

Strongest OOD combinations for your presentation
ModelDatasetWhy OODResNet-50/101eurosat-rgbTop-down satellite; no objects, no textures from trainingResNet-50/101chest X-rayGrayscale, medical, inverted contrastResNet-50/101PathMNISTMicroscopy; cellular scale, no natural scene structureEfficientNet-B4chest X-raySame as above, different architectureViT-B/16PathMNISTInteresting because ViT-21k is broader — does it help?DINOv2chest X-raySelf-supervised; does visual structure transfer without labels?CLIPPathMNISTDoes web-scale training help with microscopy? Probably not much

rm -rf ~/.cache/huggingface/